# TrendSet - Data Preprocessing

## Objective

This notebook loads, explores, cleans, and prepares the fashion retail dataset for demand forecasting and dashboard analytics.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

# Ignore unnecessary warnings
warnings.filterwarnings("ignore")

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

# Default graph size
plt.rcParams["figure.figsize"] = (12, 6)

## Load the Dataset

In [3]:
customers = pd.read_csv("../data/raw/customers.csv", low_memory=False)
discounts = pd.read_csv("../data/raw/discounts.csv")
employees = pd.read_csv("../data/raw/employees.csv")
products = pd.read_csv("../data/raw/products.csv")
stores = pd.read_csv("../data/raw/stores.csv")
transactions = pd.read_csv("../data/raw/transactions.csv")

## Dataset Overview

In [4]:
datasets = {
    "Customers": customers,
    "Discounts": discounts,
    "Employees": employees,
    "Products": products,
    "Stores": stores,
    "Transactions": transactions
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 40)
    print(f"Rows    : {df.shape[0]:,}")
    print(f"Columns : {df.shape[1]}")


Customers
----------------------------------------
Rows    : 1,643,306
Columns : 9

Discounts
----------------------------------------
Rows    : 181
Columns : 6

Employees
----------------------------------------
Rows    : 404
Columns : 4

Products
----------------------------------------
Rows    : 17,940
Columns : 12

Stores
----------------------------------------
Rows    : 35
Columns : 8

Transactions
----------------------------------------
Rows    : 6,416,827
Columns : 19


### Observations

- Transactions is the primary fact table.
- Products provides product attributes.
- Stores provides geographical information.
- Customers, Discounts and Employees are not part of the MVP.


## Data Quality Assessment

In this section, we evaluate the quality of the datasets before performing any cleaning or merging.

In [5]:
for name,df in datasets.items():
    print(f"\n{name}")
    print("-"*40)
    print(df.isnull().sum())



Customers
----------------------------------------
Customer ID           0
Name                  0
Email                 0
Telephone             0
City                  0
Country               0
Gender                0
Date Of Birth         0
Job Title        584185
dtype: int64

Discounts
----------------------------------------
Start            0
End              0
Discont          0
Description      0
Category        10
Sub Category    10
dtype: int64

Employees
----------------------------------------
Employee ID    0
Store ID       0
Name           0
Position       0
dtype: int64

Products
----------------------------------------
Product ID             0
Category               0
Sub Category           0
Description PT         0
Description DE         0
Description FR         0
Description ES         0
Description EN         0
Description ZH         0
Color              12445
Sizes               2070
Production Cost        0
dtype: int64

Stores
-----------------------------------

In [6]:
for name,df in datasets.items():
    print(f"{name}: {df.duplicated().sum()} duplicate rows")


Customers: 0 duplicate rows
Discounts: 0 duplicate rows
Employees: 0 duplicate rows
Products: 0 duplicate rows
Stores: 0 duplicate rows
Transactions: 798 duplicate rows


## Exploratory Data Analysis (EDA)

In [7]:
transactions.head()

,Invoice ID,Line,Customer ID,Product ID,Size,Color,Unit Price,Quantity,Date,Discount,Line Total,Store ID,Employee ID,Currency,Currency Symbol,SKU,Transaction Type,Payment Method,Invoice Total
0,INV-US-001-03558761,1,47162,485,M,NaN,80.5,1,2023-01-01 15:42:00,0.0,80.5,1,7,USD,$,MASU485-M-,Sale,Cash,126.7
1,INV-US-001-03558761,2,47162,2779,G,NaN,31.5,1,2023-01-01 15:42:00,0.4,18.9,1,7,USD,$,CHCO2779-G-,Sale,Cash,126.7
2,INV-US-001-03558761,3,47162,64,M,NEUTRAL,45.5,1,2023-01-01 15:42:00,0.4,27.3,1,7,USD,$,MACO64-M-NEUTRAL,Sale,Cash,126.7
3,INV-US-001-03558762,1,10142,131,M,BLUE,70.0,1,2023-01-01 20:04:00,0.4,42.0,1,6,USD,$,FECO131-M-BLUE,Sale,Cash,77.0
4,INV-US-001-03558762,2,10142,716,L,WHITE,26.0,1,2023-01-01 20:04:00,0.0,26.0,1,6,USD,$,MAT-716-L-WHITE,Sale,Cash,77.0


In [8]:
products.head()

,Product ID,Category,Sub Category,Description PT,Description DE,Description FR,Description ES,Description EN,Description ZH,Color,Sizes,Production Cost
0,1,Feminine,Coats and Blazers,Esportivo Veludo Verde Com Botões,Sport Samt Sport Mit Knöpfen,Sports Velvet Sports Avec Des Boutons,Deportes De Terciopelo Con Botones,Sports Velvet Sports With Buttons,运动天鹅绒运动与按钮,NaN,S|M|L|XL,10.73
1,2,Feminine,Sweaters and Knitwear,Luxuoso Denim Rosa Com Botões,Luxuriöser Rosa Jeans Mit Knöpfen,Léchard De Denim Rose Avec Boutons,Denim Rosa Lujoso Con Botones,Luxurious Pink Denim With Buttons,豪华的粉红色牛仔布和纽扣,PINK,S|M|L|XL,19.55
2,3,Feminine,Dresses and Jumpsuits,Retrô Tricot Preto Estampado,Black Tricot Gedruckter Tricot,Tricot Imprimé En Tricot Noir,Tricot Negro Tricot Impreso,Black Tricot Printed Tricot,黑色三角形印刷三角形,BLACK,S|M|L|XL,25.59
3,4,Feminine,Shirts and Blouses,Blusa De Algodão Básica,Basis -Baumwollbluse,Chemisier En Coton De Base,Blusa De Algodón,Basic Cotton Blouse,基本的棉衬衫,NaN,S|M|L|XL,27.62
4,5,Feminine,T-shirts and Tops,T-Shirt Básica De Algodão,Basis-Baumwoll-T-Shirt,T-Shirt En Coton De Base,Camiseta Básica De Algodón,Basic Cotton T-Shirt,基本棉T恤,NaN,S|M|L,11.69


In [9]:
stores.head()

,Store ID,Country,City,Store Name,Number of Employees,ZIP Code,Latitude,Longitude
0,1,United States,New York,Store New York,10,10001,40.7128,-74.0060
1,2,United States,Los Angeles,Store Los Angeles,8,90001,34.0522,-118.2437
2,3,United States,Chicago,Store Chicago,9,60601,41.8781,-87.6298
3,4,United States,Houston,Store Houston,10,77001,29.7604,-95.3698
4,5,United States,Phoenix,Store Phoenix,9,85001,33.4484,-112.0740


### Initial Observations

- Reviewed the first few rows of the core tables.
- Transactions will be the primary fact table.
- Products and Stores will be merged into Transactions.


## Data Merging

In [11]:
print("Transactions Columns:")
print(transactions.columns.tolist())

print("\nProducts Columns:")
print(products.columns.tolist())

print("\nStores Columns:")
print(stores.columns.tolist())


Transactions Columns:
['Invoice ID', 'Line', 'Customer ID', 'Product ID', 'Size', 'Color', 'Unit Price', 'Quantity', 'Date', 'Discount', 'Line Total', 'Store ID', 'Employee ID', 'Currency', 'Currency Symbol', 'SKU', 'Transaction Type', 'Payment Method', 'Invoice Total']

Products Columns:
['Product ID', 'Category', 'Sub Category', 'Description PT', 'Description DE', 'Description FR', 'Description ES', 'Description EN', 'Description ZH', 'Color', 'Sizes', 'Production Cost']

Stores Columns:
['Store ID', 'Country', 'City', 'Store Name', 'Number of Employees', 'ZIP Code', 'Latitude', 'Longitude']


In [12]:
# Merge transactions with products
master_df = transactions.merge(
    products,
    on="Product ID",
    how="left"
)

# Merge the result with stores
master_df = master_df.merge(
    stores,
    on="Store ID",
    how="left"
)

In [13]:
master_df.shape

(6416827, 37)

In [14]:
master_df.head()

,Invoice ID,Line,Customer ID,Product ID,Size,Color_x,Unit Price,Quantity,Date,Discount,Line Total,Store ID,Employee ID,Currency,Currency Symbol,SKU,Transaction Type,Payment Method,Invoice Total,Category,Sub Category,Description PT,Description DE,Description FR,Description ES,Description EN,Description ZH,Color_y,Sizes,Production Cost,Country,City,Store Name,Number of Employees,ZIP Code,Latitude,Longitude
0,INV-US-001-03558761,1,47162,485,M,NaN,80.5,1,2023-01-01 15:42:00,0.0,80.5,1,7,USD,$,MASU485-M-,Sale,Cash,126.7,Masculine,Suits and Blazers,Blazer Masculino De Tecido Rústico Com Textura,Männer Rustikaler Stoff Blazer Mit Textur,Blazer En Tissu Rustique Pour Hommes Avec Texture,Blazer De Tela Rústica De Los Hombres Con Textura,Men'S Rustic Fabric Blazer With Texture,男士质朴的质地西装外套,NaN,M|L|XL,12.09,United States,New York,Store New York,10,10001,40.7128,-74.006
1,INV-US-001-03558761,2,47162,2779,G,NaN,31.5,1,2023-01-01 15:42:00,0.4,18.9,1,7,USD,$,CHCO2779-G-,Sale,Cash,126.7,Children,Coats,Streetwear Seda Dourado Acolchoado,Streetwear Golden Seide,Streetwear Golden Silk,Seda Dorada,Streetwear Golden Silk,街头服装金色丝绸,NaN,P|M|G,15.14,United States,New York,Store New York,10,10001,40.7128,-74.006
2,INV-US-001-03558761,3,47162,64,M,NEUTRAL,45.5,1,2023-01-01 15:42:00,0.4,27.3,1,7,USD,$,MACO64-M-NEUTRAL,Sale,Cash,126.7,Masculine,Coats and Blazers,Luxuoso Camurça Neutro Com Capuz,Luxuriöses Neutrales Wildleder Mit Kapuze,Suede Neutre Luxueux Avec Capuche,Dulce Neutral Lujoso Con Capucha,Luxurious Neutral Suede With Hood,豪华中性绒面革带有引擎盖,NEUTRAL,M|L|XL|XXL,12.02,United States,New York,Store New York,10,10001,40.7128,-74.006
3,INV-US-001-03558762,1,10142,131,M,BLUE,70.0,1,2023-01-01 20:04:00,0.4,42.0,1,6,USD,$,FECO131-M-BLUE,Sale,Cash,77.0,Feminine,Coats and Blazers,Formal Jeans Azul Com Babados,Formelle Blaue Jeans Mit Rüschen,Jeans Bleus Formels Avec Volants,Jeans Azules Formales Con Volantes,Formal Blue Jeans With Ruffles,正式的蓝色牛仔裤和荷叶边,BLUE,S|M|L|XL,49.28,United States,New York,Store New York,10,10001,40.7128,-74.006
4,INV-US-001-03558762,2,10142,716,L,WHITE,26.0,1,2023-01-01 20:04:00,0.0,26.0,1,6,USD,$,MAT-716-L-WHITE,Sale,Cash,77.0,Masculine,T-shirts and Polos,High-Tech Camurça Branco Bordado,High-Tech-Wildleder Weiß Gestickt,En Daim De Haute Technologie Blanc Brodé,White De Gamuza De Alta Tecnología Bordada,High-Tech Suede White Embroidered,高科技绒面革白色绣花,WHITE,M|L|XL,8.31,United States,New York,Store New York,10,10001,40.7128,-74.006


In [16]:
master_df[[
    "Product ID",
    "Category",
    "Country",
    "Store Name"
]].head()

,Product ID,Category,Country,Store Name
0,485,Masculine,United States,Store New York
1,2779,Children,United States,Store New York
2,64,Masculine,United States,Store New York
3,131,Feminine,United States,Store New York
4,716,Masculine,United States,Store New York


In [17]:
print("Missing Categories:", master_df["Category"].isna().sum())
print("Missing Countries:", master_df["Country"].isna().sum())

Missing Categories: 0
Missing Countries: 0


### Merge Summary

- Successfully merged the Transactions, Products, and Stores tables.
- The resulting `master_df` will be used throughout the project for analysis, forecasting, API responses, and dashboard visualizations.

# Data Cleaning
Cleaning the merged dataset before feature engineering.

In [18]:
# Shape before cleaning
print("Before:", master_df.shape)

# Remove duplicate rows
master_df = master_df.drop_duplicates()

print("After:", master_df.shape)

Before: (6416827, 37)
After: (6416029, 37)


In [19]:
master_df.isnull().sum().sort_values(ascending=False).head(15)

Color              4350231
Color_product      4350231
Sizes               413049
Size                413049
Invoice ID               0
Description DE           0
Description FR           0
Description ES           0
Description EN           0
Description ZH           0
Country                  0
Production Cost          0
Sub Category             0
City                     0
Store Name               0
dtype: int64

In [20]:
master_df.dtypes

Invoice ID              object
Line                     int64
Customer ID              int64
Product ID               int64
Size                    object
Color                   object
Unit Price             float64
Quantity                 int64
Date                    object
Discount               float64
Line Total             float64
Store ID                 int64
Employee ID              int64
Currency                object
Currency Symbol         object
SKU                     object
Transaction Type        object
Payment Method          object
Invoice Total          float64
Category                object
Sub Category            object
Description PT          object
Description DE          object
Description FR          object
Description ES          object
Description EN          object
Description ZH          object
Color_product           object
Sizes                   object
Production Cost        float64
Country                 object
City                    object
Store Na

In [21]:
master_df["Date"] = pd.to_datetime(master_df["Date"])

In [22]:
master_df.dtypes[["Date"]]

Date    datetime64[ns]
dtype: object

# Feature Engineering

Creating new features that will later be used for analytics, forecasting, and dashboard visualizations.

In [23]:
# Date-based features

master_df["Year"] = master_df["Date"].dt.year
master_df["Month"] = master_df["Date"].dt.month
master_df["Month Name"] = master_df["Date"].dt.month_name()
master_df["Day"] = master_df["Date"].dt.day
master_df["Weekday"] = master_df["Date"].dt.day_name()
master_df["Quarter"] = master_df["Date"].dt.quarter

In [24]:
master_df[[
    "Date",
    "Year",
    "Month",
    "Month Name",
    "Weekday",
    "Quarter"
]].head()

,Date,Year,Month,Month Name,Weekday,Quarter
0,2023-01-01 15:42:00,2023,1,January,Sunday,1
1,2023-01-01 15:42:00,2023,1,January,Sunday,1
2,2023-01-01 15:42:00,2023,1,January,Sunday,1
3,2023-01-01 20:04:00,2023,1,January,Sunday,1
4,2023-01-01 20:04:00,2023,1,January,Sunday,1


In [25]:
# Profit calculation

master_df["Profit"] = (
    master_df["Line Total"] -
    master_df["Production Cost"]
)

In [26]:
master_df[[
    "Line Total",
    "Production Cost",
    "Profit"
]].head()

,Line Total,Production Cost,Profit
0,80.5,12.09,68.41
1,18.9,15.14,3.76
2,27.3,12.02,15.28
3,42.0,49.28,-7.28
4,26.0,8.31,17.69


In [27]:
# High-level statistics

master_df[[
    "Unit Price",
    "Line Total",
    "Production Cost",
    "Profit"
]].describe()

,Unit Price,Line Total,Production Cost,Profit
count,6.416029e+06,6.416029e+06,6.416029e+06,6.416029e+06
mean,1.324614e+02,1.142215e+02,1.733211e+01,9.688938e+01
std,1.850947e+02,2.115725e+02,1.204930e+01,2.093507e+02
min,2.000000e+00,-3.348000e+03,5.600000e-01,-3.416110e+03
25%,3.250000e+01,2.475000e+01,8.650000e+00,1.133000e+01
50%,5.100000e+01,4.350000e+01,1.432000e+01,2.658000e+01
75%,1.165000e+02,1.090000e+02,2.264000e+01,8.416000e+01
max,1.153500e+03,3.460500e+03,7.719000e+01,3.415010e+03


## Feature Engineering Summary

New features created:

- Year
- Month
- Month Name
- Day
- Weekday
- Quarter
- Profit

These features will support demand forecasting, business analytics, and dashboard visualizations.

# Save Processed Dataset

Saving the cleaned master dataset for future analytics, forecasting, and backend APIs.

In [28]:
master_df.to_csv("../data/processed/master_dataset.csv", index=False)

print("Dataset saved successfully!")

Dataset saved successfully!


In [29]:
master_df.shape

(6416029, 44)

In [30]:
saved_df = pd.read_csv("../data/processed/master_dataset.csv")

saved_df.head()

,Invoice ID,Line,Customer ID,Product ID,Size,Color,Unit Price,Quantity,Date,Discount,Line Total,Store ID,Employee ID,Currency,Currency Symbol,SKU,Transaction Type,Payment Method,Invoice Total,Category,Sub Category,Description PT,Description DE,Description FR,Description ES,Description EN,Description ZH,Color_product,Sizes,Production Cost,Country,City,Store Name,Number of Employees,ZIP Code,Latitude,Longitude,Year,Month,Month Name,Day,Weekday,Quarter,Profit
0,INV-US-001-03558761,1,47162,485,M,NaN,80.5,1,2023-01-01 15:42:00,0.0,80.5,1,7,USD,$,MASU485-M-,Sale,Cash,126.7,Masculine,Suits and Blazers,Blazer Masculino De Tecido Rústico Com Textura,Männer Rustikaler Stoff Blazer Mit Textur,Blazer En Tissu Rustique Pour Hommes Avec Texture,Blazer De Tela Rústica De Los Hombres Con Textura,Men'S Rustic Fabric Blazer With Texture,男士质朴的质地西装外套,NaN,M|L|XL,12.09,United States,New York,Store New York,10,10001,40.7128,-74.006,2023,1,January,1,Sunday,1,68.41
1,INV-US-001-03558761,2,47162,2779,G,NaN,31.5,1,2023-01-01 15:42:00,0.4,18.9,1,7,USD,$,CHCO2779-G-,Sale,Cash,126.7,Children,Coats,Streetwear Seda Dourado Acolchoado,Streetwear Golden Seide,Streetwear Golden Silk,Seda Dorada,Streetwear Golden Silk,街头服装金色丝绸,NaN,P|M|G,15.14,United States,New York,Store New York,10,10001,40.7128,-74.006,2023,1,January,1,Sunday,1,3.76
2,INV-US-001-03558761,3,47162,64,M,NEUTRAL,45.5,1,2023-01-01 15:42:00,0.4,27.3,1,7,USD,$,MACO64-M-NEUTRAL,Sale,Cash,126.7,Masculine,Coats and Blazers,Luxuoso Camurça Neutro Com Capuz,Luxuriöses Neutrales Wildleder Mit Kapuze,Suede Neutre Luxueux Avec Capuche,Dulce Neutral Lujoso Con Capucha,Luxurious Neutral Suede With Hood,豪华中性绒面革带有引擎盖,NEUTRAL,M|L|XL|XXL,12.02,United States,New York,Store New York,10,10001,40.7128,-74.006,2023,1,January,1,Sunday,1,15.28
3,INV-US-001-03558762,1,10142,131,M,BLUE,70.0,1,2023-01-01 20:04:00,0.4,42.0,1,6,USD,$,FECO131-M-BLUE,Sale,Cash,77.0,Feminine,Coats and Blazers,Formal Jeans Azul Com Babados,Formelle Blaue Jeans Mit Rüschen,Jeans Bleus Formels Avec Volants,Jeans Azules Formales Con Volantes,Formal Blue Jeans With Ruffles,正式的蓝色牛仔裤和荷叶边,BLUE,S|M|L|XL,49.28,United States,New York,Store New York,10,10001,40.7128,-74.006,2023,1,January,1,Sunday,1,-7.28
4,INV-US-001-03558762,2,10142,716,L,WHITE,26.0,1,2023-01-01 20:04:00,0.0,26.0,1,6,USD,$,MAT-716-L-WHITE,Sale,Cash,77.0,Masculine,T-shirts and Polos,High-Tech Camurça Branco Bordado,High-Tech-Wildleder Weiß Gestickt,En Daim De Haute Technologie Blanc Brodé,White De Gamuza De Alta Tecnología Bordada,High-Tech Suede White Embroidered,高科技绒面革白色绣花,WHITE,M|L|XL,8.31,United States,New York,Store New York,10,10001,40.7128,-74.006,2023,1,January,1,Sunday,1,17.69


# Day 2 Summary

Completed:
- Loaded all datasets
- Explored the data
- Assessed data quality
- Merged Products and Stores into Transactions
- Removed duplicate rows
- Converted Date column to datetime
- Created new time-based features
- Calculated Profit
- Saved the cleaned master dataset

Output:
`data/processed/master_dataset.csv`